# 188. H₂O：怎样用 Heavy Hitter + Recent 策略淘汰 KV Cache？

> **面试问题：注意力累计分数、recent window、heavy-hitter budget 与因果顺序怎样实现，为什么不能只保留最新 token？**

## 先给结论

这题的关键不是调用框架，而是把状态、预算、验证器和可复放的制品合同显式化。教学代码用小数据验证不变量；生产实现仍需替换为真实模型、内核、隔离环境与线上观测。

## 一手资料

- [H2O](https://arxiv.org/abs/2306.14048)
- [StreamingLLM](https://arxiv.org/abs/2309.17453)
- [PagedAttention](https://arxiv.org/abs/2309.06180)


In [ ]:
import hashlib  # 导入本单元依赖。
import json  # 导入本单元依赖。
import math  # 导入本单元依赖。
from dataclasses import asdict, dataclass  # 导入本单元依赖。
CAPACITY = 4  # 计算并保存当前中间结果。
RECENT = 2  # 计算并保存当前中间结果。
assert CAPACITY > RECENT  # 用断言验证关键不变量。
assert RECENT > 0  # 用断言验证关键不变量。
assert CAPACITY == 4  # 用断言验证关键不变量。


## 1. 最小状态与输入合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def update_scores(scores, attention):  # 定义可复用的核心函数。
    if len(scores) != len(attention):  # 按条件选择控制路径。
        raise ValueError("分数与注意力长度不一致")  # 非法输入立即显式失败。
    return [old + new for old, new in zip(scores, attention)]  # 返回当前计算结果。
assert update_scores([1, 0], [0.5, 2]) == [1.5, 2]  # 用断言验证关键不变量。
assert update_scores([], []) == []  # 用断言验证关键不变量。
assert update_scores([0], [1])[0] == 1  # 用断言验证关键不变量。


## 2. 核心公式或状态转换

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def h2o_keep(scores, capacity, recent):  # 定义可复用的核心函数。
    if capacity < recent or recent < 0:  # 按条件选择控制路径。
        raise ValueError("预算非法")  # 非法输入立即显式失败。
    n = len(scores)  # 计算并保存当前中间结果。
    recent_ids = list(range(max(0, n - recent), n))  # 计算并保存当前中间结果。
    heavy = sorted(range(max(0, n - recent)), key=lambda i: (-scores[i], i))[:max(0, capacity - len(recent_ids))]  # 计算并保存当前中间结果。
    return sorted(set(heavy + recent_ids))  # 返回当前计算结果。
keep = h2o_keep([9, 1, 8, 2, 3, 4], 4, 2)  # 计算并保存当前中间结果。
assert keep == [0, 2, 4, 5]  # 用断言验证关键不变量。
assert h2o_keep([1, 2], 4, 2) == [0, 1]  # 用断言验证关键不变量。
assert h2o_keep([], 4, 2) == []  # 用断言验证关键不变量。


## 3. 候选选择与验证

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass  # 计算并保存当前中间结果。
class H2OCache:  # 定义保存状态的数据结构。
    tokens: list  # 计算并保存当前中间结果。
    scores: list  # 计算并保存当前中间结果。
    def append(self, token, attention):  # 定义可复用的核心函数。
        self.scores = update_scores(self.scores, attention)  # 计算并保存当前中间结果。
        self.tokens.append(token)  # 计算并保存当前中间结果。
        self.scores.append(0.0)  # 计算并保存当前中间结果。
    def evict(self, capacity, recent):  # 定义可复用的核心函数。
        ids = h2o_keep(self.scores, capacity, recent)  # 计算并保存当前中间结果。
        self.tokens = [self.tokens[i] for i in ids]  # 计算并保存当前中间结果。
        self.scores = [self.scores[i] for i in ids]  # 计算并保存当前中间结果。
cache = H2OCache(["a", "b", "c", "d"], [9, 1, 8, 2])  # 计算并保存当前中间结果。
cache.evict(3, 1)  # 计算并保存当前中间结果。
assert cache.tokens == ["a", "c", "d"]  # 用断言验证关键不变量。
assert len(cache.scores) == 3  # 用断言验证关键不变量。
assert cache.tokens[-1] == "d"  # 用断言验证关键不变量。


## 4. 主路径实现

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
cache.append("e", [1, 1, 1])  # 计算并保存当前中间结果。
assert cache.tokens[-1] == "e"  # 用断言验证关键不变量。
assert len(cache.tokens) == len(cache.scores)  # 用断言验证关键不变量。
cache.evict(3, 1)  # 计算并保存当前中间结果。
assert len(cache.tokens) <= 3  # 用断言验证关键不变量。


## 5. 边界与失败分支

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def retention_report(before, after):  # 定义可复用的核心函数。
    return {"kept": len(after), "evicted": len(before) - len(after), "ratio": len(after) / max(len(before), 1)}  # 返回当前计算结果。
report = retention_report(list(range(8)), cache.tokens)  # 计算并保存当前中间结果。
assert report["kept"] <= 3  # 用断言验证关键不变量。
assert report["evicted"] >= 0  # 用断言验证关键不变量。
assert 0 <= report["ratio"] <= 1  # 用断言验证关键不变量。


## 6. 正确性与基线对照

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def causal_order(indices):  # 定义可复用的核心函数。
    return indices == sorted(indices) and len(indices) == len(set(indices))  # 返回当前计算结果。
assert causal_order(h2o_keep([2, 1, 3], 2, 1))  # 用断言验证关键不变量。
assert not causal_order([2, 1])  # 用断言验证关键不变量。
assert causal_order([])  # 用断言验证关键不变量。


## 7. 成本或数据合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass(frozen=True)  # 计算并保存当前中间结果。
class H2OArtifact:  # 定义保存状态的数据结构。
    capacity: int  # 计算并保存当前中间结果。
    recent: int  # 计算并保存当前中间结果。
    score_rule: str  # 计算并保存当前中间结果。
artifact = H2OArtifact(4, 2, "cumulative-attention")  # 计算并保存当前中间结果。
assert artifact.capacity > artifact.recent  # 用断言验证关键不变量。
assert artifact.score_rule.startswith("cumulative")  # 用断言验证关键不变量。
assert len(hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()) == 64  # 用断言验证关键不变量。


## 8. 制品版本与面试收束

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
assert 0 in h2o_keep([100, 0, 0, 0], 2, 1)  # 用断言验证关键不变量。
assert 3 in h2o_keep([100, 0, 0, 0], 2, 1)  # 用断言验证关键不变量。
assert h2o_keep([1], 1, 1) == [0]  # 用断言验证关键不变量。


## 面试收束

回答时按目标、状态合同、核心算法、失败分支、评测指标与发布版本组织；受控例子只证明实现不变量，不代表真实模型或生产系统性能。
